In [ ]:
# ============================================================
# БЛОК 4: FEATURE ENGINEERING — ДОБАВЛЕНИЕ НОВЫХ ПРИЗНАКОВ
# Проект: Прогнозирование объема вкладов населения РФ
# Автор: Надежда Силкина
# Дата: 2026
# ============================================================

# ============================================================
# 1. ПОДКЛЮЧЕНИЕ БИБЛИОТЕК
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Настройка графиков
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Библиотеки загружены")

# ============================================================
# 2. ЗАГРУЗКА ДАННЫХ И ПОДГОТОВКА БАЗОВОЙ МОДЕЛИ
# ============================================================

url = 'https://raw.githubusercontent.com/HopeSilkina/deposits_forecast_project/main/data/processed_deposits_data.xlsx'
df = pd.read_excel(url, sheet_name='data')
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

# Создание базовых признаков (лаги DEPOS)
df['DEPOS_log'] = np.log(df['DEPOS'])
for lag in [1, 3, 6, 12]:
    df[f'DEPOS_lag_{lag}'] = df['DEPOS'].shift(lag)

# Подготовка X и y
X_base = df.drop(['DEPOS', 'DEPOS_log'], axis=1).dropna()
y = df.loc[X_base.index, 'DEPOS']

# Разделение на train/test
train_size = len(X_base) - 12
X_train_base, X_test_base = X_base.iloc[:train_size], X_base.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

# Масштабирование для базовой модели
scaler_base = StandardScaler()
X_train_scaled_base = scaler_base.fit_transform(X_train_base)
X_test_scaled_base = scaler_base.transform(X_test_base)

# Базовая модель Ridge
ridge_base = Ridge(alpha=1.0)
ridge_base.fit(X_train_scaled_base, y_train)
y_pred_base = ridge_base.predict(X_test_scaled_base)

r2_base = r2_score(y_test, y_pred_base)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_base))
mae_base = mean_absolute_error(y_test, y_pred_base)

print("="*60)
print("БАЗОВАЯ МОДЕЛЬ (RIDGE, ALPHA=1.0)")
print("="*60)
print(f"R² = {r2_base:.4f}")
print(f"RMSE = {rmse_base:.2f} млрд руб.")
print(f"MAE = {mae_base:.2f} млрд руб.")

# ============================================================
# 3. ДОБАВЛЕНИЕ НОВЫХ ПРИЗНАКОВ
# ============================================================

print("\n" + "="*60)
print("ДОБАВЛЕНИЕ НОВЫХ ПРИЗНАКОВ")
print("="*60)

# Копируем данные для новых признаков
df_feat = df.copy()

# 3.1. Сезонные фиктивные переменные
print("\n🔍 Добавление сезонных фиктивных переменных...")
df_feat['Month'] = df_feat.index.month
month_dummies = pd.get_dummies(df_feat['Month'], prefix='month', drop_first=True)
df_feat = pd.concat([df_feat, month_dummies], axis=1)
print(f"   Добавлено 11 сезонных переменных")

# 3.2. Фиктивная переменная post_2022
print("\n🔍 Добавление фиктивной переменной post_2022...")
df_feat['post_2022'] = (df_feat.index >= '2023-01-01').astype(int)
print(f"   post_2022 = 1 для {df_feat['post_2022'].sum()} записей (с 2023 года)")

# 3.3. Фиктивная переменная covid
print("\n🔍 Добавление фиктивной переменной covid...")
df_feat['covid'] = ((df_feat.index >= '2020-03-01') & (df_feat.index <= '2022-01-01')).astype(int)
print(f"   covid = 1 для {df_feat['covid'].sum()} записей")

# 3.4. Фиктивная переменная regime_cred1 (по DEPOS)
print("\n🔍 Добавление фиктивной переменной regime_cred1...")
threshold_depos = 32000
df_feat['regime_cred1'] = (df_feat['DEPOS'] > threshold_depos).astype(int)
print(f"   regime_cred1 = 1 для {df_feat['regime_cred1'].sum()} записей (DEPOS > {threshold_depos})")

# 3.5. Фиктивная переменная anomaly_wage (с адаптивным порогом)
print("\n🔍 Добавление фиктивной переменной anomaly_wage с адаптивным порогом...")

# Функция для проверки влияния anomaly_wage с заданным порогом
def test_anomaly_threshold(thresh, df_feat, X_train_base, X_test_base, y_train, y_test, r2_base):
    # Расчет аномалий
    model_wage = LinearRegression()
    model_wage.fit(df_feat[['WAGE']].values, df_feat['DEPOS'].values)
    df_feat['residual_wage'] = df_feat['DEPOS'] - model_wage.predict(df_feat[['WAGE']].values)
    threshold_anomaly = thresh * df_feat['residual_wage'].std()
    df_feat['anomaly_wage'] = (df_feat['residual_wage'] < threshold_anomaly).astype(int)

    # Создаем X с новым признаком
    X_train_temp = X_train_base.copy()
    X_test_temp = X_test_base.copy()
    X_train_temp['anomaly_wage'] = df_feat.loc[X_train_temp.index, 'anomaly_wage']
    X_test_temp['anomaly_wage'] = df_feat.loc[X_test_temp.index, 'anomaly_wage']

    # Масштабируем (вместе с новым признаком)
    scaler_temp = StandardScaler()
    X_train_scaled_temp = scaler_temp.fit_transform(X_train_temp)
    X_test_scaled_temp = scaler_temp.transform(X_test_temp)

    # Обучаем модель
    ridge_temp = Ridge(alpha=1.0)
    ridge_temp.fit(X_train_scaled_temp, y_train)
    y_pred_temp = ridge_temp.predict(X_test_scaled_temp)
    r2_temp = r2_score(y_test, y_pred_temp)

    return r2_temp, df_feat['anomaly_wage'].sum()

# Пробуем разные пороги
thresholds = [-1.0, -1.2, -1.5, -2.0]
best_r2_anomaly = r2_base
best_threshold = None
best_anomaly_count = 0

for thresh in thresholds:
    r2_temp, count = test_anomaly_threshold(thresh, df_feat, X_train_base, X_test_base, y_train, y_test, r2_base)
    print(f"   Порог {thresh}σ: аномалий = {count}, R² = {r2_temp:.4f}")

    if r2_temp > best_r2_anomaly:
        best_r2_anomaly = r2_temp
        best_threshold = thresh
        best_anomaly_count = count

# Сохраняем лучшую версию anomaly_wage
if best_threshold is not None:
    model_wage = LinearRegression()
    model_wage.fit(df_feat[['WAGE']].values, df_feat['DEPOS'].values)
    df_feat['residual_wage'] = df_feat['DEPOS'] - model_wage.predict(df_feat[['WAGE']].values)
    threshold_anomaly_best = best_threshold * df_feat['residual_wage'].std()
    df_feat['anomaly_wage'] = (df_feat['residual_wage'] < threshold_anomaly_best).astype(int)
    print(f"\n   ✅ Лучший порог: {best_threshold}σ (аномалий: {best_anomaly_count})")
    print(f"   ✅ Улучшение R²: {best_r2_anomaly - r2_base:.4f}")

# 3.6. Лаги для WAGE, CPI, USDind
print("\n🔍 Добавление лагов для WAGE, CPI, USDind...")
for col in ['WAGE', 'CPI', 'USDind']:
    for lag in [1, 3, 6]:
        df_feat[f'{col}_lag_{lag}'] = df_feat[col].shift(lag)
print(f"   Добавлено 9 лаговых признаков (3 переменных × 3 лага)")

# 3.7. Взаимодействие UNEM и DEP1
print("\n🔍 Добавление взаимодействия UNEM × DEP1...")
df_feat['UNEM_DEP1'] = df_feat['UNEM'] * df_feat['DEP1']
print("   Добавлено взаимодействие UNEM × DEP1")

# ============================================================
# 4. ПОДГОТОВКА ДАННЫХ ДЛЯ МОДЕЛИ С НОВЫМИ ПРИЗНАКАМИ
# ============================================================

print("\n" + "="*60)
print("ПОДГОТОВКА ДАННЫХ ДЛЯ МОДЕЛИ С НОВЫМИ ПРИЗНАКАМИ")
print("="*60)

# Удаляем вспомогательные столбцы
X_new = df_feat.drop(['DEPOS', 'DEPOS_log', 'Month', 'residual_wage'], axis=1).dropna()

print(f"📊 Число признаков в базовой модели: {X_base.shape[1]}")
print(f"📊 Число признаков в новой модели: {X_new.shape[1]}")
print(f"📊 Добавлено признаков: {X_new.shape[1] - X_base.shape[1]}")

# Разделение на train/test
X_train_new, X_test_new = X_new.iloc[:train_size], X_new.iloc[train_size:]

# Масштабирование
scaler_new = StandardScaler()
X_train_scaled_new = scaler_new.fit_transform(X_train_new)
X_test_scaled_new = scaler_new.transform(X_test_new)

# ============================================================
# 5. ОБУЧЕНИЕ МОДЕЛИ С НОВЫМИ ПРИЗНАКАМИ
# ============================================================

print("\n" + "="*60)
print("ОБУЧЕНИЕ МОДЕЛИ С НОВЫМИ ПРИЗНАКАМИ")
print("="*60)

ridge_new = Ridge(alpha=1.0)
ridge_new.fit(X_train_scaled_new, y_train)
y_pred_new = ridge_new.predict(X_test_scaled_new)

r2_new = r2_score(y_test, y_pred_new)
rmse_new = np.sqrt(mean_squared_error(y_test, y_pred_new))
mae_new = mean_absolute_error(y_test, y_pred_new)

print(f"\n📊 Результаты модели с новыми признаками:")
print(f"   R² = {r2_new:.4f}")
print(f"   RMSE = {rmse_new:.2f} млрд руб.")
print(f"   MAE = {mae_new:.2f} млрд руб.")

# ============================================================
# 6. СРАВНЕНИЕ МОДЕЛЕЙ
# ============================================================

print("\n" + "="*60)
print("СРАВНЕНИЕ МОДЕЛЕЙ")
print("="*60)

comparison = pd.DataFrame({
    'Модель': ['Базовая (Ridge)', 'С новыми признаками'],
    'R²': [r2_base, r2_new],
    'RMSE': [rmse_base, rmse_new],
    'MAE': [mae_base, mae_new],
    'Признаков': [X_base.shape[1], X_new.shape[1]]
})

print(comparison.to_string(index=False))

improvement = r2_new - r2_base
print(f"\n📊 Улучшение R²: {improvement:.4f}")

if improvement > 0.01:
    print("   ✅ Новые признаки ЗНАЧИТЕЛЬНО улучшают модель")
elif improvement > 0:
    print("   ✅ Новые признаки НЕМНОГО улучшают модель")
else:
    print("   ℹ️ Новые признаки НЕ улучшают модель")

# ============================================================
# 7. АНАЛИЗ КАЖДОГО НОВОГО ПРИЗНАКА
# ============================================================

print("\n" + "="*60)
print("АНАЛИЗ КАЖДОГО НОВОГО ПРИЗНАКА")
print("="*60)

def test_feature(X_train_base, X_test_base, feature_name, y_train, y_test, r2_base):
    X_train_test = X_train_base.copy()
    X_test_test = X_test_base.copy()
    X_train_test[feature_name] = X_train_new[feature_name]
    X_test_test[feature_name] = X_test_new[feature_name]

    scaler_test = StandardScaler()
    X_train_scaled_test = scaler_test.fit_transform(X_train_test)
    X_test_scaled_test = scaler_test.transform(X_test_test)

    ridge_test = Ridge(alpha=1.0)
    ridge_test.fit(X_train_scaled_test, y_train)
    y_pred_test = ridge_test.predict(X_test_scaled_test)
    r2_test = r2_score(y_test, y_pred_test)
    improvement = r2_test - r2_base
    return improvement

new_features = [
    'month_2', 'month_3', 'month_4', 'month_5', 'month_6',
    'month_7', 'month_8', 'month_9', 'month_10', 'month_11', 'month_12',
    'post_2022', 'covid', 'regime_cred1', 'anomaly_wage',
    'WAGE_lag_1', 'WAGE_lag_3', 'WAGE_lag_6',
    'CPI_lag_1', 'CPI_lag_3', 'CPI_lag_6',
    'USDind_lag_1', 'USDind_lag_3', 'USDind_lag_6',
    'UNEM_DEP1'
]

print("\n🔍 Влияние каждого нового признака на R²:")
results = []
for feature in new_features:
    imp = test_feature(X_train_base, X_test_base, feature, y_train, y_test, r2_base)
    results.append({'Признак': feature, 'Улучшение R²': imp})
    status = '✅' if imp > 0.001 else 'ℹ️'
    print(f"   {status} {feature}: {imp:.4f}")

results_df = pd.DataFrame(results).sort_values('Улучшение R²', ascending=False)
print("\n📊 Топ-5 признаков по улучшению:")
print(results_df.head(5).to_string(index=False))

# ============================================================
# 8. ИТОГОВЫЙ ВЫВОД
# ============================================================

print("\n" + "="*60)
print("📌 ИТОГОВЫЙ ВЫВОД ПО БЛОКУ 4")
print("="*60)

best_r2 = max(r2_base, r2_new)
best_model = 'Базовая' if r2_base >= r2_new else 'С новыми признаками'

print(f"\n🏆 Лучшая модель: {best_model} (R² = {best_r2:.4f})")

if improvement > 0.01:
    print("\n✅ Добавление новых признаков ЗНАЧИТЕЛЬНО улучшает модель")
elif improvement > 0:
    print("\n✅ Добавление новых признаков НЕМНОГО улучшает модель")
else:
    print("\nℹ️ Добавление новых признаков НЕ улучшает модель")

print("\n📌 КЛЮЧЕВЫЕ ВЫВОДЫ:")
print("   1. Наибольшее улучшение дали признаки: " +
      ", ".join(results_df.head(3)['Признак'].tolist()))
print("   2. Сезонные переменные " +
      ("улучшают" if any('month' in r['Признак'] and r['Улучшение R²'] > 0.001 for r in results) else "не улучшают") + " модель")
print("   3. Фиктивная переменная post_2022 " +
      ("улучшает" if results_df[results_df['Признак'] == 'post_2022']['Улучшение R²'].values[0] > 0.001 else "не улучшает") + " модель")
print("   4. Лаги для WAGE, CPI, USDind " +
      ("улучшают" if any('_lag_' in r['Признак'] and r['Улучшение R²'] > 0.001 for r in results) else "не улучшают") + " модель")

print("\n✅ Блок 4 завершен")